# Which Tool, When?

You've now built agent systems at five levels: the **raw loop**, **PydanticAI**, **LangGraph**, a **custom harness** around the loop, and — in the sister course — **n8n**. This closing chapter is the decision framework: given a real task, which do you reach for? The honest answer is usually "the simplest one that works," and this notebook makes that concrete.

There's no new framework code here — it's the map you take with you, followed by one integration challenge that proves the pieces connect.

## Four defaults, plus a harness you own

| | Raw loop (Block 0) | PydanticAI (Block 1) | LangGraph (Block 2) | n8n (no-code) |
|--|--------------------|----------------------|---------------------|---------------|
| **You write** | ~40 lines of Python | functions + decorators | nodes + edges + state | drag nodes on a canvas |
| **Best at** | learning; total control | one clean, typed agent | explicit state, cycles, multi-agent | fast integrations, non-coders |
| **Flow is** | fully visible, yours | implicit (hidden loop) | an explicit, drawable graph | a visual canvas |
| **Typed output / DI** | by hand | first-class | via LangChain | limited |
| **Persistence / HITL** | by hand | add-on / manual | built-in (checkpointers, interrupt) | built-in nodes |
| **Integrations** | you write them | tools / MCP | tools / MCP | hundreds built in |
| **Weakest at** | reinventing everything | deep stateful orchestration | boilerplate for simple agents | code-level control |

A **custom harness** is not the default fifth framework. It is what you build when the product must own boundaries that a framework does not: provider adapters, a permission and event model, model-facing history filtering, custom execution, or crash-safe external effects (0.9, 32c). It can wrap PydanticAI or LangGraph rather than replace them. The test is not "do I want control?"; it is "which concrete requirement can the existing runtime not express safely?"

## A decision flow

```
Is the task a fixed sequence you can draw in advance?
   ├─ yes → it's a WORKFLOW, not an agent. Write plain Python (0.4). Don't reach for a framework.
   └─ no  → it needs an agent. Then:

        Do non-coders need to build/own it, or is it mostly wiring SaaS together?
           ├─ yes → n8n (the no-code course).
           └─ no  → you're in code. Then:

                Does it need explicit state, cycles, durable HITL, or multiple agents?
                   ├─ no  → PydanticAI. One clean, typed agent. (Most apps live here.)
                   └─ yes → LangGraph. Use checkpointers (2.3), interrupt (2.4),
                            cycles (2.5), and multi-agent supervision (2.6).

Then test the application boundary:
   ├─ Framework lifecycle and persistence cover it → keep the framework.
   └─ You need a custom permission/event model, executor, provider boundary, or effect contract
      → own that harness layer (0.9, 32c); the framework can still run inside it.

Cross-cutting choices:
   • Need standard remote tool integration? Use MCP with whichever runtime you chose.
   • Touching an external system? Add idempotency at that boundary; a checkpoint is not proof
     that an email, payment, or Slack message happened exactly once.

And always: build the loop by hand ONCE (Block 0) so you can debug whatever you choose.
```

## Practice: choose and justify

Run the flowchart on five real briefs. For each, pick **one** primary approach and write a one-sentence justification — the justification is the actual skill. Try before opening the solutions.

1. Classify thousands of support tickets a day into five fixed categories.
2. Approve every piece of generated SQL before it runs against the production database.
3. A non-technical ops team wants a daily Slack digest stitched from three SaaS tools.
4. Answer ad-hoc questions about arbitrary CSVs a user uploads.
5. Hand a customer conversation to the right billing, tech, or HR specialist.

::::{dropdown} 🛠️ Solutions (one defensible answer each)
:color: secondary

1. **Raw structured output / PydanticAI, no agent loop.** It's a fixed classification — a single typed call (1.3), not an agent. Reaching for LangGraph here is the "don't build an agent" anti-pattern (0.4).
2. **LangGraph plus an application-owned effect boundary.** `interrupt()` and a checkpointer give durable pause/resume before the SQL runs (2.4). If approval leads to an external write, key that effect and record its outcome as in 32c; resuming graph state alone does not make the write exactly-once.
3. **n8n.** The value *is* the scheduled trigger plus prebuilt SaaS integrations — a non-coder should own it. Writing this in Python would reinvent nodes n8n ships (the sister course).
4. **A data/coding agent with sandboxed code execution** (3.2). One general "run code" tool beats a hundred narrow ones — but approval is not containment, so use a real sandbox for untrusted code.
5. **PydanticAI if the routing is one-shot; a LangGraph supervisor/handoff if it's a running conversation** (2.6). Don't add multi-agent machinery unless the conversation genuinely spans specialists.

MCP may supply tools in any of these code solutions; it does not decide the control flow. The pattern across all five is to name the *one* capability the task truly needs, pick the simplest runtime that has it, then design the external-effect boundary separately.
::::

## The recurring lesson

Every block of this course repeated the same warning, and it is the main thing to take away:

- **Reach for the simplest row that solves the problem** (0.0). A single LLM call or a plain-Python workflow beats an agent whenever the steps are known.
- **Frameworks remove boilerplate, not understanding** (1.0). You built the loop by hand so you can debug PydanticAI and LangGraph — because you know what they're doing underneath.
- **Multi-agent is not automatically better** (2.6). One good agent with good tools usually wins.
- **Measure, don't eyeball** (1.6). Non-deterministic software needs evals, not vibes.
- **State recovery is not effect recovery** (2.9, 3.2c). Checkpoints restore computation; idempotency and delivery contracts protect the outside world.

The 2026 bar for agent engineers isn't "can you make a demo" — it's *judgment*: knowing which tool a problem needs, which boundary your application must still own, and when **not** to use an agent at all. That judgment is the point of the course.

## A map of the wider ecosystem

You'll meet other frameworks, and the concepts you built transfer directly — so you can read them without getting lost. **OpenAI Agents SDK** centers hosted tools and explicit handoffs; **CrewAI** models role-based teams of agents; **smolagents** (Hugging Face) keeps code-executing agents tiny; the **Claude Agent SDK** exposes the coding-agent loop you dissected in 0.7–0.8. They're all the same primitives — a loop, tools, typed I/O, memory — with different ergonomics.

The thing to weigh is **churn cost**: every extra framework buys some convenience but adds another API surface to learn, version, and debug. That's why this course taught you the loop by hand first. *Keep the concepts; rent the frameworks* — and switch only when a specific one removes real pain, not because it's new.

## Final integration challenge

::::{dropdown} 🛠️ Ship one course project end to end
:color: secondary

Choose the Data Analyst (32), Smart Onboarding (32b), or Atlas (32c):

1. Expose its main operation through the FastAPI pattern from 30.
2. Connect the server-side Gradio client from 31; keep the agent credential out of the browser.
3. Add one authenticated-user rate limit and one smoke test.
4. Run the project's existing eval or trajectory assertions against the deployed path.

**Done when:** a fresh client can call the deployed project, a bad credential fails closed, the browser contains no server secret, and a recorded test proves both the API response and the project's core invariant. That is the difference between three separate notebook demos and one system you can defend in an interview.
::::

## Where this goes next

Once that integration works, deepen the parts your target role needs:

- **MCP at scale** — scopes, server discovery, audit logs, and many tool servers.
- **Deployment hardening** — real user auth, durable databases, queues, concurrency tests, cost budgets, and incident handling.
- **Durable long-term memory** — replace the in-memory Store from 28 and define retention, deletion, and tenant isolation.
- **Evaluation-driven development** — version the eval set, run it in CI, and iterate from failure categories.
- **Frontier** — browser/computer-use agents, voice agents, and higher-level orchestration frameworks; reasonable to skip until a product needs them.

You have the foundation the rest of it builds on. Go ship something measurable.